# Tornado API - Data Preperation
> - The Tornado API's data comes from one CSV before being uploaded into the MongoDB Documents 
> - This script handles all of the data collection, cleaning and transformation for this project

In [1]:
import pandas as pd 
import numpy as np 
import re
import tornado_util_functions as utils

tornado_util_functions loaded...


## 1. Data Collection


### 1.1. Tornado Details

In [6]:
def get_tornado_details(storm_details_csv):
    storm_details_return = storm_details_csv[storm_details_csv["EVENT_TYPE"] == "Tornado"]
    storm_details_return = storm_details_return[[
        "EPISODE_ID", #ID assigned by NWS to denote the storm episode; links to location & fatality files
        "EVENT_ID", #ID assigned by NWS to note a single, small part that goes into a specific storm episode; links to location & fatality files
        "STATE", # The state name where the event occurred
        "CZ_FIPS", # The county FIPS number, a unique number assigned to the country by NIST or NWS Forecast Zone Number
        "CZ_NAME", # The name of the County
        "WFO", # NWS Forecast Office's area of responsibility (County Warning Area) in which the event occurred
        "BEGIN_DATE_TIME", # Date and time the event began: MM/DD/YYYY 24 Hour Time AM/PM
        "CZ_TIMEZONE", # Time Zone for the County
        "END_DATE_TIME", # Date and time the event ended: MM/DD/YYYY 24 Hour Time AM/PM
        "INJURIES_DIRECT", # The number of injuries directly related to the weather event
        "INJURIES_INDIRECT", # The number of injuries indirectly related to the weather event
        "DEATHS_DIRECT", # The number of deaths directly related to the weather event
        "DEATHS_INDIRECT", # The number of deaths indirectly related to the weather event
        "DAMAGE_PROPERTY", # The estimated amount of damage to property incurred by the weather event
        "DAMAGE_CROPS", # The estimated amount of damage to crop incurred by the weather event
        "SOURCE", # The source reporting the weather event (Trained Spotter, Storm Chaser, Law Enforcement etc.)
        "TOR_F_SCALE", # The F or EF Scale that describes the strength of the tornado based on the amount and type of damange caused by the tornado
        "TOR_LENGTH", # The length of the tornado or tornado segment while on the ground
        "TOR_WIDTH", # Width of the tornado or tornado segment while on the ground
        "TOR_OTHER_WFO", # Indicates the continuation of a Tornado as it crossed from one NWS Forecast Office to another. The subsequent WFO identifier is provided within this field
        "TOR_OTHER_CZ_STATE", # The two character representation for the state name of the continuing tornado segment as it crossed from one county or zone to another. The subsequent 2-Letter State ID is provide.
        "TOR_OTHER_CZ_FIPS", # The FIPS number of the county entered by the continuing tornado segment as it crossed from one county to another.  The subsequent FIPS number is provided within this field. 
        "TOR_OTHER_CZ_NAME", # The FIPS name of the county entered by the continuing tornado segment as it crossed from one county to another.  The subsequent county or zone name is provided within this field in ALL CAPS. 
        "BEGIN_LAT", # The latitude in decimal degrees of the begin point of the event or damage path. 
        "BEGIN_LON", # The longitude in decimal degrees of the begin point of the event or damage path. 
        "END_LAT", # The latitude in decimal degrees of the end point of the event or damage path.
        "END_LON", # The longitude in decimal degrees of the end point of the event or damage path.
        "EPISODE_NARRATIVE", # The episode narrative depicting the general nature and overall activity of the episode.  The narrative is created by NWS.
        "EVENT_NARRATIVE" # The event narrative provides more specific details of the individual event. The event narrative is provided by NWS
    ]]

    storm_details_return.rename(columns={
        'EPISODE_ID': 'episode_id',
        'EVENT_ID': 'event_id',
        'STATE': 'state',
        'CZ_FIPS': 'county_fips',
        'CZ_NAME': 'county_name',
        'WFO': 'wfo',
        'BEGIN_DATE_TIME': 'begin_date_time',
        'CZ_TIMEZONE': 'county_timezone',
        'END_DATE_TIME': 'end_date_time',
        'INJURIES_INDIRECT': 'injuries_indirect',
        'INJURIES_DIRECT': 'injuries_direct',
        'DEATHS_DIRECT': 'deaths_direct',
        'DEATHS_INDIRECT': 'deaths_indirect',
        'DAMAGE_PROPERTY': 'property_damage',
        'DAMAGE_CROPS': 'crop_damage',
        'SOURCE': 'source',
        'TOR_F_SCALE': 'tornado_rating',
        'TOR_LENGTH': 'tornado_length',
        'TOR_WIDTH': 'tornado_max_width',
        'TOR_OTHER_WFO': 'other_wfos',
        'TOR_OTHER_CZ_STATE': 'other_states',
        'TOR_OTHER_CZ_FIPS': 'other_counties_fips',
        'TOR_OTHER_CZ_NAME': 'other_counties_names',
        'BEGIN_LAT': 'begin_latitude',
        'BEGIN_LON': 'begin_longitude',
        'END_LAT': 'end_latitude',
        'END_LON': 'end_longitude',
        'EPISODE_NARRATIVE': 'episode_narrative',
        'EVENT_NARRATIVE': 'event_narrative'
    }, inplace=True)
    return storm_details_return

In [ ]:
storm_details_2025 = pd.read_csv('imported_data/storm_details/StormEvents_details-ftp_v1.0_d2025_c20260323.csv')


tornado_details_2025 = get_tornado_details(storm_details_2025)

tornado_details_2025.head()

,episode_id,event_id,state,county_fips,county_name,wfo,begin_date_time,county_timezone,end_date_time,injuries_direct,...,other_wfos,other_states,other_counties_fips,other_counties_names,begin_latitude,begin_longitude,end_latitude,end_longitude,episode_narrative,event_narrative
1,200337,1241136,MICHIGAN,27,CASS,IWX,30-MAR-25 15:52:00,EST-5,30-MAR-25 15:55:00,0,...,NaN,NaN,NaN,NaN,41.7900,-86.1000,41.8200,-86.0700,A cold front pushed into the area during the a...,A brief EF-1 tornado was confirmed in Edwardsb...
43,201725,1253359,KANSAS,47,EDWARDS,DDC,18-MAY-25 18:21:00,CST-6,18-MAY-25 18:33:00,0,...,NaN,NaN,NaN,NaN,37.8900,-99.3700,37.9500,-99.3600,A regional outbreak of tornadoes occurred acro...,Fairly short-lived tornado. Video of tornado f...
49,201006,1249677,TENNESSEE,69,HARDEMAN,MEG,15-MAR-25 06:56:00,CST-6,15-MAR-25 07:02:00,0,...,NaN,NaN,NaN,NaN,35.0966,-89.0021,35.1106,-88.9581,An upper low over the Central Plains moved int...,A weak tornado developed near Callahan Road li...
51,201005,1249680,MISSISSIPPI,95,MONROE,MEG,15-MAR-25 12:16:00,CST-6,15-MAR-25 12:18:00,0,...,NaN,NaN,NaN,NaN,33.9719,-88.5998,33.9902,-88.5726,An upper low over the Central Plains moved int...,A brief tornado developed near the intersectio...
53,201003,1249588,ARKANSAS,31,CRAIGHEAD,MEG,14-MAR-25 23:16:00,CST-6,14-MAR-25 23:17:00,0,...,MEG,AR,55.0,GREENE,35.9601,-90.6996,35.9670,-90.6860,An upper low over the Central Plains moved int...,A tornado touched down in northern Craighead C...


In [8]:
storm_details_1950 = pd.read_csv('imported_data/storm_details/StormEvents_details-ftp_v1.0_d1950_c20260323.csv')
tornado_details_1950 = get_tornado_details(storm_details_1950)
tornado_details_1950.head()

,episode_id,event_id,state,county_fips,county_name,wfo,begin_date_time,county_timezone,end_date_time,injuries_direct,...,other_wfos,other_states,other_counties_fips,other_counties_names,begin_latitude,begin_longitude,end_latitude,end_longitude,episode_narrative,event_narrative
0,NaN,10096222,OKLAHOMA,149,WASHITA,NaN,28-APR-50 14:45:00,CST,28-APR-50 14:45:00,0,...,NaN,NaN,NaN,NaN,35.12,-99.20,35.17,-99.20,NaN,NaN
1,NaN,10120412,TEXAS,93,COMANCHE,NaN,29-APR-50 15:30:00,CST,29-APR-50 15:30:00,0,...,NaN,NaN,NaN,NaN,31.90,-98.60,31.73,-98.60,NaN,NaN
2,NaN,10104927,PENNSYLVANIA,77,LEHIGH,NaN,05-JUL-50 18:00:00,CST,05-JUL-50 18:00:00,2,...,NaN,NaN,NaN,NaN,40.58,-75.70,40.65,-75.47,NaN,NaN
3,NaN,10104928,PENNSYLVANIA,43,DAUPHIN,NaN,05-JUL-50 18:30:00,CST,05-JUL-50 18:30:00,0,...,NaN,NaN,NaN,NaN,40.60,-76.75,NaN,NaN,NaN,NaN
4,NaN,10104929,PENNSYLVANIA,39,CRAWFORD,NaN,24-JUL-50 14:40:00,CST,24-JUL-50 14:40:00,0,...,NaN,NaN,NaN,NaN,41.63,-79.68,NaN,NaN,NaN,NaN
